# AFib detector — robustness to noise and signal-quality gating (Notebook 4)
**Bektur Asilov · COMSE-23 · Ala-Too**

Notebooks 2–3 showed that most false alarms come from noisy signal (on CinC 2017 the model called 51.7 % of the
"too noisy" recordings AF). Here real ECG noise from the **MIT-BIH Noise Stress Test Database** (electrode motion
`em`, muscle artefact `ma`, baseline wander `bw`) is added at controlled levels to the CinC 2017 AF and Normal
recordings the model has never seen, and two versions of the system are compared:

* **without gating** — every 32-beat decision counts (as in Notebook 3);
* **with gating** — a decision is discarded when the signal-quality index (SQI) of the 10 s before it is *poor*
  (< 0.4). This is the **same SQI the app and the server already show** (`backend/app/ecgtools.py: quality`):
  flat line, plausible heart rate, QRS-shape consistency, high-frequency noise. A recording with no decision left
  gets **no verdict** ("signal too poor") instead of a guess.

Fixed before the run: model, threshold 0.5, majority rule per recording (as in Notebook 3), SQI threshold 0.4,
SNR levels 24/12/6/0 dB defined as signal power / noise power after the 0.5–40 Hz band-pass, random noise segment
per recording with a fixed seed. The same 3,328 recordings (537 AF + 2,791 Normal with ≥ 33 beats) are used at every level.

> Kaggle: *Add Input* → `marufhs/atrial-fibrillation`; Internet **On** (CinC 2017 and NSTDB from the PhysioNet open-data mirror).

In [ ]:
import os, glob, time, random, contextlib, io, json, subprocess, sys, zipfile, urllib.request
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.signal import butter, sosfilt, resample_poly
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'wfdb'], check=True)
import wfdb
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import confusion_matrix, roc_auc_score

SEED = 42
FS_ORIG, FS = 250, 125
N_RR, RR_STRIDE, DEV_STRIDE = 32, 8, 4
OUT_DIR = os.environ.get('OUT_DIR', '/kaggle/working')
LABELS = {'(N': 0, '(AFIB': 1}
tf.get_logger().setLevel('ERROR')
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

AFDB_DIR = os.environ.get('AFDB_DIR') or [d for d in {os.path.dirname(p) for p in glob.glob('/kaggle/input/**/04015.dat', recursive=True)} if '/old' not in d][0]
records = sorted(r[:-4] for r in os.listdir(AFDB_DIR) if r.endswith('.dat') and os.path.exists(os.path.join(AFDB_DIR, r[:-4] + '.atr')))
print(len(records), 'AFDB records')

In [ ]:
MA = 19; DECAY = 0.5 ** (1 / (2.0 * FS)); THR, REFR, WARM = 0.35, int(0.25 * FS), 2 * FS
try:
    from numba import njit
except ImportError:
    njit = lambda f: f

@njit
def detect_r(x):
    out = np.empty(len(x) // 20 + 10, np.int64); n = 0
    buf = np.zeros(MA); ssum = 0.0; prev = x[0]; pk = 0.0; last = -10**9; armed = True
    for i in range(len(x)):
        d = x[i] - prev; prev = x[i]; e = d * d
        ssum += e - buf[i % MA]; buf[i % MA] = e
        env = ssum / MA; pk = max(pk * DECAY, env)
        if i < WARM: continue
        if armed and env > THR * pk and i - last > REFR:
            out[n] = i; n += 1; last = i; armed = False
            if n >= len(out): break
        elif not armed and env < 0.5 * THR * pk:
            armed = True
    return out[:n]

def rr_features(rr):
    rr = np.clip(rr, 0.2, 2.0)
    return np.stack([(rr - 0.8) / 0.25, (rr / rr.mean() - 1) / 0.15], -1).astype(np.float32)

def to_125(sig, fs):
    # same band-pass as the firmware (0.5-40 Hz, 2nd order), then to 125 Hz
    x = sosfilt(butter(2, [0.5, 40], btype='band', fs=fs, output='sos'), np.nan_to_num(sig))
    if fs == 250: return x[::2]
    from math import gcd
    g = gcd(int(fs), FS); return resample_poly(x, FS // g, int(fs) // g)

In [ ]:
def tcn_block(x, f, k, d, drop):
    sc = x
    x = layers.Conv1D(f, k, padding='causal', dilation_rate=d, activation='relu')(x)
    x = layers.Dropout(drop)(x)
    x = layers.Conv1D(f, k, padding='causal', dilation_rate=d)(x)
    if sc.shape[-1] != f: sc = layers.Conv1D(f, 1)(sc)
    return layers.ReLU()(layers.Add()([x, sc]))

def build_rr_tcn():
    inp = keras.Input(shape=(N_RR, 2)); x = inp
    for d in (1, 2, 4, 8): x = tcn_block(x, 16, 3, d, 0.1)
    return keras.Model(inp, layers.Dense(1, activation='sigmoid')(layers.GlobalAveragePooling1D()(x)))

def to_int8(m, Xrep):
    conv = tf.lite.TFLiteConverter.from_keras_model(m)
    conv.optimizations = [tf.lite.Optimize.DEFAULT]
    rep = Xrep[np.random.RandomState(0).choice(len(Xrep), min(1000, len(Xrep)), replace=False)]
    conv.representative_dataset = lambda: ([s[None]] for s in rep)
    conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    conv.inference_input_type = tf.int8; conv.inference_output_type = tf.int8
    with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
        return conv.convert()

def run_tflite(blob, Xd, bs=4096):
    outs = []
    it = tf.lite.Interpreter(model_content=blob)
    for i in range(0, len(Xd), bs):
        xb = Xd[i:i + bs]; inp = it.get_input_details()[0]
        it.resize_tensor_input(inp['index'], [len(xb), N_RR, 2]); it.allocate_tensors()
        inp, out = it.get_input_details()[0], it.get_output_details()[0]
        sc, zp = inp['quantization']; it.set_tensor(inp['index'], np.clip(np.round(xb / sc + zp), -128, 127).astype(np.int8))
        it.invoke(); sc, zp = out['quantization']
        outs.append(((it.get_tensor(out['index']).astype(np.float32) - zp) * sc).ravel())
    return np.concatenate(outs) if outs else np.zeros(0)

PREP = {}
for rec in records:
    sig = wfdb.rdrecord(os.path.join(AFDB_DIR, rec), channels=[0]).p_signal[:, 0]
    ann = wfdb.rdann(os.path.join(AFDB_DIR, rec), 'atr')
    starts = list(ann.sample); ends = starts[1:] + [len(sig)]
    segs = [(s, e, LABELS.get(a.strip().rstrip('\x00'))) for s, e, a in zip(starts, ends, ann.aux_note)]
    x = to_125(sig, FS_ORIG); beats = detect_r(x); rr = np.diff(beats) / FS
    lab_at = np.full(len(x), -1, np.int8)
    for s, e, l in segs:
        if l is not None: lab_at[s // 2:e // 2] = l
    Xw, yw = [], []
    for k in range(0, len(rr) - N_RR + 1, RR_STRIDE):
        labs = lab_at[beats[k]:beats[k + N_RR]]
        if len(labs) == 0 or labs.min() < 0 or labs.min() != labs.max(): continue
        Xw.append(rr_features(rr[k:k + N_RR])); yw.append(int(labs[0]))
    PREP[rec] = (np.array(Xw, np.float32).reshape(-1, N_RR, 2), np.array(yw, np.int32))

rng = np.random.RandomState(SEED)
for _ in range(500):   # 2 patients only for early stopping, both rhythms present (same rule as Notebook 2)
    val = list(rng.choice(records, 2, replace=False))
    if all(sum((PREP[v][1] == c).sum() for v in val) >= 100 for c in (0, 1)): break
train = [r for r in records if r not in val]
Xtr = np.concatenate([PREP[r][0] for r in train]); ytr = np.concatenate([PREP[r][1] for r in train])
Xva = np.concatenate([PREP[r][0] for r in val]);   yva = np.concatenate([PREP[r][1] for r in val])
cw = {0: len(ytr) / (2 * (ytr == 0).sum()), 1: len(ytr) / (2 * ytr.sum())}
model = build_rr_tcn(); model.compile(optimizer=keras.optimizers.Adam(1e-3), loss='binary_crossentropy')
ds = tf.data.Dataset.from_tensor_slices((Xtr, ytr)).shuffle(50000, seed=SEED).batch(256).prefetch(2)
h = model.fit(ds, validation_data=(Xva, yva), epochs=int(os.environ.get('EPOCHS', 40)), class_weight=cw, verbose=0,
              callbacks=[keras.callbacks.EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True),
                         keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2)])
BLOB = to_int8(model, Xtr)
open(f'{OUT_DIR}/afib_rr_tcn_all23_int8.tflite', 'wb').write(BLOB)
print('trained on', len(train), 'patients (+2 for early stopping:', val, '), epochs', len(h.history['loss']),
      '| windows', len(ytr), '| int8 model', len(BLOB), 'bytes')

## Signal-quality index (numpy port of `ecgtools.quality`, same formula and thresholds)

In [ ]:
def _hp(x, fs):
    w = max(3, int(0.6 * fs)) | 1; h = w // 2
    c = np.concatenate([[0.0], np.cumsum(x)]); n = len(x); i = np.arange(n)
    a = np.maximum(0, i - h); b = np.minimum(n, i + h + 1)
    return x - (c[b] - c[a]) / (b - a)

def sqi(x, peaks, fs=FS):
    n = len(x)
    if n < 2 * fs: return 0.0
    y = _hp(np.asarray(x, float), fs)
    s_flat = 0.0 if (y.max() - y.min()) < 0.05 or (np.abs(np.diff(y)) < 1e-4).sum() / n > 0.5 else 1.0
    bpm = len(peaks) / (n / fs) * 60; s_rate = 1.0 if 30 <= bpm <= 200 else 0.0
    h = int(0.1 * fs); segs = np.array([y[p - h:p + h] for p in peaks if p - h >= 0 and p + h <= n])
    if len(segs) >= 3:
        t = np.median(segs, 0); s = segs - segs.mean(1, keepdims=True); tc = t - t.mean()
        r = (s @ tc) / (np.sqrt((s ** 2).sum(1) * (tc ** 2).sum()) + 1e-12); s_shape = max(0.0, float(r.mean()))
    else:
        s_shape = 0.0
    d2 = y[2:] - 2 * y[1:-1] + y[:-2]; noise = np.sqrt((d2 ** 2).mean()) / (y.std() or 1e-12)
    s_noise = min(1.0, max(0.0, 1.5 - noise))
    return round(min(1.0, max(0.0, s_flat * s_rate * (0.6 * s_shape + 0.4 * s_noise))), 2)
SQI_POOR = 0.4

## CinC 2017 recordings and NSTDB noise

In [ ]:
CINC_DIR = os.environ.get('CINC_DIR')
if not CINC_DIR:
    # the PhysioNet open-data mirror on AWS is much faster than physionet.org from Kaggle; same files
    MIRRORS = ['https://physionet-open.s3.amazonaws.com/challenge-2017/1.0.0/',
               'https://physionet.org/files/challenge-2017/1.0.0/']
    os.makedirs('/kaggle/tmp', exist_ok=True)
    for f in ('REFERENCE-v3.csv', 'training2017.zip'):
        dst = f'/kaggle/tmp/{f}'
        for base in MIRRORS:
            try:
                t = time.time(); subprocess.run(['curl', '-fsSL', '--max-time', '1200', '-o', dst, base + f], check=True)
                print(f, os.path.getsize(dst) // 1024, 'KB from', base, f'in {time.time()-t:.0f} s', flush=True); break
            except Exception as e:
                print('mirror failed:', base, e, flush=True)
    zipfile.ZipFile('/kaggle/tmp/training2017.zip').extractall('/kaggle/tmp')
    CINC_DIR = '/kaggle/tmp/training2017'
    REF = '/kaggle/tmp/REFERENCE-v3.csv'
else:
    REF = os.path.join(CINC_DIR, 'REFERENCE-v3.csv')
ref = pd.read_csv(REF, header=None, names=['record', 'label'])
print(len(ref), 'recordings;', ref.label.value_counts().to_dict())


NS_DIR = os.environ.get('NS_DIR')
if not NS_DIR:
    NS_DIR = '/kaggle/tmp/nstdb'; os.makedirs(NS_DIR, exist_ok=True)
    for rec in ('em', 'ma', 'bw'):
        for ext in ('hea', 'dat'):
            subprocess.run(['curl', '-fsSL', '-o', f'{NS_DIR}/{rec}.{ext}',
                            f'https://physionet-open.s3.amazonaws.com/nstdb/1.0.0/{rec}.{ext}'], check=True)
NOISE = {}
for rec in ('em', 'ma', 'bw'):
    r = wfdb.rdrecord(os.path.join(NS_DIR, rec), channels=[0]); NOISE[rec] = (r.p_signal[:, 0], r.fs)
print({k: (len(v[0]), v[1]) for k, v in NOISE.items()})

SIG = {}   # the recordings used everywhere: AF and Normal with >= 33 beats on the clean signal
for rec, lab in zip(ref.record, ref.label):
    if lab not in ('A', 'N'): continue
    r = wfdb.rdrecord(os.path.join(CINC_DIR, rec)); x = r.p_signal[:, 0]
    if len(detect_r(to_125(x, r.fs))) >= N_RR + 1: SIG[rec] = (x, r.fs, lab)
print(len(SIG), 'recordings;', pd.Series([v[2] for v in SIG.values()]).value_counts().to_dict())

## Run every noise type and level, with and without gating

In [ ]:
from scipy.signal import resample_poly as _rp
def noise_for(kind, n, fs, rng):
    z, nfs = NOISE[kind]
    if nfs != fs:
        from math import gcd
        g = gcd(int(fs), int(nfs)); z = _rp(z, int(fs) // g, int(nfs) // g)
    s = rng.randint(0, len(z) - n); return z[s:s + n]

def bandpass(x, fs): return sosfilt(butter(2, [0.5, 40], btype='band', fs=fs, output='sos'), x)

def evaluate(x, fs):
    x125 = to_125(x, fs); beats = detect_r(x125); rr = np.diff(beats) / FS
    ks = list(range(0, len(rr) - N_RR + 1, DEV_STRIDE))
    if not ks: return dict(n_dec=0)
    p = run_tflite(BLOB, np.array([rr_features(rr[k:k + N_RR]) for k in ks], np.float32))
    keep = []
    for k in ks:
        end = beats[k + N_RR]; a = max(0, end - 10 * FS)
        seg = x125[a:end]; pk = beats[(beats >= a) & (beats < end)] - a
        keep.append(sqi(seg, pk) >= SQI_POOR)
    keep = np.array(keep)
    out = dict(n_dec=len(ks), frac_af=float((p >= .5).mean()), n_keep=int(keep.sum()))
    if keep.any(): out['frac_af_gated'] = float((p[keep] >= .5).mean())
    return out

CONDS = [('clean', None)] + [(k, s) for k in ('em', 'ma', 'bw') for s in (24, 12, 6, 0)]
rows, T0 = [], time.time()
for kind, snr in CONDS:
    rng = np.random.RandomState(SEED)
    for rec, (x, fs, lab) in SIG.items():
        if kind == 'clean':
            xn = x
        else:
            z = noise_for(kind, len(x), fs, rng)
            ps, pn = bandpass(x, fs).var(), bandpass(z, fs).var()
            xn = x + z * np.sqrt(ps / (pn * 10 ** (snr / 10)) if pn > 0 else 0)
        rows.append(dict(record=rec, label=lab, noise=kind, snr=snr if snr is not None else 99, **evaluate(xn, fs)))
    print(f'{kind:5s} {snr}  [{time.time()-T0:.0f} s]', flush=True)
D = pd.DataFrame(rows); D.to_csv(f'{OUT_DIR}/noise_predictions.csv', index=False)

## Results

In [ ]:
def metrics(g):
    y = (g.label == 'A').values
    pred = (g.frac_af.fillna(0) >= .5).values
    has = g.frac_af_gated.notna().values
    predg = (g.frac_af_gated.fillna(0) >= .5).values
    return pd.Series(dict(
        sens=pred[y].mean(), spec=(~pred[~y]).mean(),
        sens_gated=predg[y & has].mean() if (y & has).any() else np.nan,
        spec_gated=(~predg[~y & has]).mean() if (~y & has).any() else np.nan,
        no_verdict_AF=(~has[y]).mean(), no_verdict_N=(~has[~y]).mean(),
        false_AF_calls_N=pred[~y].sum(), false_AF_calls_N_gated=(predg & has)[~y].sum(),
        missed_AF_gated=(y & has & ~predg).sum() + 0, AF_without_verdict=(y & ~has).sum() + 0))
T = D.groupby(['noise', 'snr']).apply(metrics).reset_index().sort_values(['noise', 'snr'], ascending=[True, False])
T.to_csv(f'{OUT_DIR}/noise_summary.csv', index=False)
pd.set_option('display.width', 200); print(T.round(3).to_string(index=False))
json.dump(T.round(4).to_dict(orient='records'), open(f'{OUT_DIR}/noise_summary.json', 'w'), indent=1, default=float)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(17, 4.2))
for kind, c in (('em', '#C0362F'), ('ma', '#E0A030'), ('bw', '#5B8DEF')):
    t = T[T.noise == kind].sort_values('snr'); cl = T[T.noise == 'clean'].iloc[0]
    xs = list(t.snr) + [30]
    ax[0].plot(xs, list(t.spec * 100) + [cl.spec * 100], 'o--', c=c, alpha=.5, label=f'{kind} no gating')
    ax[0].plot(xs, list(t.spec_gated * 100) + [cl.spec_gated * 100], 'o-', c=c, label=f'{kind} with SQI gating')
    ax[1].plot(xs, list(t.sens * 100) + [cl.sens * 100], 'o--', c=c, alpha=.5)
    ax[1].plot(xs, list(t.sens_gated * 100) + [cl.sens_gated * 100], 'o-', c=c)
    ax[2].plot(xs, list(t.no_verdict_N * 100) + [cl.no_verdict_N * 100], 'o-', c=c, label=kind)
for a, tt in zip(ax, ('Specificity (Normal not called AF), %', 'Sensitivity, %', 'Normal recordings with no verdict, %')):
    a.set_title(tt); a.set_xlabel('SNR, dB  (30 = clean)'); a.grid(alpha=.3)
ax[0].legend(fontsize=8); ax[2].legend()
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/noise_results.png', dpi=150); plt.show()
print('total', round((time.time() - T0) / 60, 1), 'min')